In [22]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import time
import matplotlib.pyplot as plt

In [23]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU is available and will be used: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("GPU not found. The model will run on the CPU.")

GPU is available and will be used: NVIDIA GeForce RTX 4060


In [24]:
DATA_FOLDER = '../Weather_Merged_CSVs'
PLOT_FOLDER = 'GRU_Plots'
os.makedirs(PLOT_FOLDER, exist_ok=True)

In [25]:
n_steps_to_test = [7, 14, 21, 30, 60]
all_crops_summary = []

In [26]:
def create_sequences(data, n_steps, target_column):
    X, y = [], []
    for i in range(len(data) - n_steps):
        X.append(data.iloc[i:(i + n_steps)].values)
        y.append(data.iloc[i + n_steps][target_column])
    return np.array(X), np.array(y).reshape(-1, 1)

In [27]:
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout_prob):
        super(GRUModel, self).__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout_prob if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout_prob)
        self.fc = nn.Linear(hidden_size, output_size)
    def forward(self, x):
        out, _ = self.gru(x)
        out = self.dropout(out[:, -1, :])
        out = self.fc(out)
        return out

In [ ]:
csv_files = [f for f in os.listdir(DATA_FOLDER) if f.endswith('.csv')]

for csv_file in csv_files:
    try:
        crop_name = os.path.splitext(csv_file)[0]
        file_path = os.path.join(DATA_FOLDER, csv_file)
        
        print("\n" + "="*70)
        print(f"Processing Crop: {crop_name}")
        print("="*70)
        df = pd.read_csv(file_path)
        df['Price Date'] = pd.to_datetime(df['Price Date'])
        df.set_index('Price Date', inplace=True)
        df.sort_index(inplace=True)
        
        if len(df) < 100: # Skip very small files
            print(f"Skipping {crop_name} due to insufficient data ({len(df)} rows).")
            continue

        # Feature Engineering & Encoding
        df['day_of_year'] = df.index.dayofyear
        df['week_of_year'] = df.index.isocalendar().week.astype(int)
        df['month'] = df.index.month

        categorical_cols = ['District Name', 'Market Name', 'Commodity', 'Variety', 'Grade']
        for col in categorical_cols:
            if df[col].dtype == 'object':
                encoder = LabelEncoder()
                df[col] = encoder.fit_transform(df[col])

        # Scaling
        scaler = MinMaxScaler(feature_range=(0, 1))
        scaled_data = scaler.fit_transform(df)
        df_scaled = pd.DataFrame(scaled_data, columns=df.columns)
        target_column = 'Modal Price (Rs./Quintal)'

        # --- Hyperparameter Tuning Loop for the current crop ---
        crop_results = {}
        best_nrmse_for_crop = float('inf')
        best_n_steps_for_crop = -1
        best_predictions_for_crop = None
        best_actuals_for_crop = None

        for n_steps in n_steps_to_test:
            print(f"\n--- Testing {crop_name} with n_steps = {n_steps} ---")
            
            if len(df_scaled) <= n_steps:
                print(f"Skipping n_steps={n_steps} as it's too large for the dataset size.")
                continue

            # (The inner loop logic is the same as your original notebook)
            X, y = create_sequences(df_scaled, n_steps, target_column)
            split = int(0.8 * len(X))
            X_train, X_test = X[split:], X[split:]
            y_train, y_test = y[split:], y[split:]
            X_train_tensor = torch.from_numpy(X_train).float()
            y_train_tensor = torch.from_numpy(y_train).float()
            X_test_tensor = torch.from_numpy(X_test).float()
            y_test_tensor = torch.from_numpy(y_test).float()
            train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
            test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
            train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
            test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            input_size = X_train.shape[2]
            model = GRUModel(input_size, hidden_size=50, num_layers=2, output_size=1, dropout_prob=0.2).to(device)
            criterion = nn.MSELoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
            num_epochs = 50
            for epoch in range(num_epochs):
                model.train()
                for batch_X, batch_y in train_loader:
                    batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                    outputs = model(batch_X)
                    loss = criterion(outputs, batch_y)
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
            
            model.eval()
            all_predictions = []
            with torch.no_grad():
                for batch_X, _ in test_loader:
                    batch_X = batch_X.to(device)
                    outputs = model(batch_X)
                    all_predictions.append(outputs.cpu().numpy())
            predictions = np.concatenate(all_predictions)

            price_col_index = df_scaled.columns.get_loc(target_column)
            dummy_pred = np.zeros((len(predictions), df_scaled.shape[1]))
            dummy_pred[:, price_col_index] = predictions.flatten()
            inversed_predictions = scaler.inverse_transform(dummy_pred)[:, price_col_index]
            dummy_actual = np.zeros((len(y_test), df_scaled.shape[1]))
            dummy_actual[:, price_col_index] = y_test.flatten()
            inversed_actual = scaler.inverse_transform(dummy_actual)[:, price_col_index]
            
            rmse = np.sqrt(mean_squared_error(inversed_actual, inversed_predictions))
            mean_actual_price = np.mean(inversed_actual)
            nrmse = rmse / mean_actual_price if mean_actual_price != 0 else 0
            
            print(f"n_steps = {n_steps} | RMSE = {rmse:.2f} | nRMSE = {nrmse:.4f}")

            if nrmse < best_nrmse_for_crop:
                best_nrmse_for_crop = nrmse
                best_n_steps_for_crop = n_steps
                best_predictions_for_crop = inversed_predictions
                best_actuals_for_crop = inversed_actual

        # --- After tuning for the current crop, save its best results ---
        if best_n_steps_for_crop != -1:
            best_rmse = np.sqrt(mean_squared_error(best_actuals_for_crop, best_predictions_for_crop))
            summary = {
                'Crop': crop_name,
                'Best n_steps': best_n_steps_for_crop,
                'Best RMSE': round(best_rmse, 2),
                'Best nRMSE (%)': round(best_nrmse_for_crop * 100, 2)
            }
            all_crops_summary.append(summary)

            # Generate and save the plot for the best model of this crop
            plt.figure(figsize=(14, 7))
            plt.plot(best_actuals_for_crop, color='red', label='Actual Price')
            plt.plot(best_predictions_for_crop, color='blue', label=f'Predicted Price (Best n_steps={best_n_steps_for_crop})')
            plt.title(f'Best Model Prediction for {crop_name}')
            plt.xlabel('Time (Test Set)')
            plt.ylabel('Price (Rs./Quintal)')
            plt.legend()
            plot_filename = os.path.join(PLOT_FOLDER, f"{crop_name}_prediction_plot.png")
            plt.savefig(plot_filename)
            plt.close() # Close the plot to free up memory
            print(f"\nSaved best plot for {crop_name} to {plot_filename}")

    except Exception as e:
        print(f"\nCould not process {csv_file}. Error: {e}")
        continue


Processing Crop: Bajra-2015-2019

Could not process Bajra-2015-2019.csv. Error: 'RangeIndex' object has no attribute 'dayofyear'

Processing Crop: Bajra-2019-2022
Skipping Bajra-2019-2022 due to insufficient data (55 rows).

Processing Crop: Bajra-2022-2025

Could not process Bajra-2022-2025.csv. Error: 'RangeIndex' object has no attribute 'dayofyear'

Processing Crop: Banana - Green

Could not process Banana - Green.csv. Error: 'RangeIndex' object has no attribute 'dayofyear'

Processing Crop: Banana

Could not process Banana.csv. Error: 'RangeIndex' object has no attribute 'dayofyear'

Processing Crop: Blackgram-2015-2019
Skipping Blackgram-2015-2019 due to insufficient data (84 rows).

Processing Crop: Blackgram-2019-2022
Skipping Blackgram-2019-2022 due to insufficient data (75 rows).

Processing Crop: Blackgram-2022-2025

Could not process Blackgram-2022-2025.csv. Error: 'RangeIndex' object has no attribute 'dayofyear'

Processing Crop: Cashewnuts

Could not process Cashewnuts.cs

In [29]:
summary_df = pd.DataFrame(all_crops_summary)
summary_df.sort_values(by='Best nRMSE (%)', inplace=True)
summary_filename = 'all_crops_best_results.csv'
summary_df.to_csv(summary_filename, index=False)

print("\n\n" + "="*70)
print(f"Processing complete. Summary of results saved to '{summary_filename}'")
print("="*70)
print(summary_df)

KeyError: 'Best nRMSE (%)'